Connect SQLite3 to pandas to get data.

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect('../../market_data.db')

In [2]:
sales_df = pd.read_sql("SELECT * FROM sales_history", conn)
competitor_df = pd.read_sql("SELECT * FROM competitor_intelligence", conn)
inventory_df = pd.read_sql("SELECT * FROM inventory_cost", conn)

In [3]:
sales_df.head()
competitor_df.head()
inventory_df.head()



,product_id,current_stock,base_cost
0,WIDGET-A,100,10.0
1,GADGET-B,50,25.0
2,SENSOR-C,200,5.0


Merge all three tables and Start of Day-5


In [4]:
df=sales_df.merge(
    competitor_df,
    on='product_id',
    how='left'
).merge(
    inventory_df,
    on='product_id',
    how='left'
)
df.head()

,sale_id,date,product_id,units_sold,price,promotion_status,intel_id,competitor_price,timestamp,current_stock,base_cost
0,1,2026-01-01,WIDGET-A,50,15.0,None,1,18.5,2026-02-19 18:26:52,100,10.0
1,2,2026-01-08,WIDGET-A,30,20.0,None,1,18.5,2026-02-19 18:26:52,100,10.0
2,3,2026-01-01,GADGET-B,20,40.0,None,2,35.0,2026-02-19 18:26:52,50,25.0
3,4,2026-01-08,GADGET-B,22,38.0,Discount,2,35.0,2026-02-19 18:26:52,50,25.0


Sort the values by product ID and Date.

In [5]:
df['date']=pd.to_datetime(df['date'])
df=df.sort_values(by=['product_id','date'])
df.head()

,sale_id,date,product_id,units_sold,price,promotion_status,intel_id,competitor_price,timestamp,current_stock,base_cost
2,3,2026-01-01,GADGET-B,20,40.0,None,2,35.0,2026-02-19 18:26:52,50,25.0
3,4,2026-01-08,GADGET-B,22,38.0,Discount,2,35.0,2026-02-19 18:26:52,50,25.0
0,1,2026-01-01,WIDGET-A,50,15.0,None,1,18.5,2026-02-19 18:26:52,100,10.0
1,2,2026-01-08,WIDGET-A,30,20.0,None,1,18.5,2026-02-19 18:26:52,100,10.0


New Column Price Gap and Price Gap Percentage.


In [6]:
df['price_gap']=df['price']-df['competitor_price']
df['price_gap_percentage']=round((df['price']-df['competitor_price'])/df['competitor_price']*100,2)
df.head()

,sale_id,date,product_id,units_sold,price,promotion_status,intel_id,competitor_price,timestamp,current_stock,base_cost,price_gap,price_gap_percentage
2,3,2026-01-01,GADGET-B,20,40.0,None,2,35.0,2026-02-19 18:26:52,50,25.0,5.0,14.29
3,4,2026-01-08,GADGET-B,22,38.0,Discount,2,35.0,2026-02-19 18:26:52,50,25.0,3.0,8.57
0,1,2026-01-01,WIDGET-A,50,15.0,None,1,18.5,2026-02-19 18:26:52,100,10.0,-3.5,-18.92
1,2,2026-01-08,WIDGET-A,30,20.0,None,1,18.5,2026-02-19 18:26:52,100,10.0,1.5,8.11


New Column : Inventory Velocity